# Observations and output sites

Define the onshore wave observations available for your case (buoys, tide gauges, spectral sites, ...), preview them on a map, and save a **JSON (GeoJSON format)** file compatible with QGIS and the HyGloW config loader.

Workflow:

1. Set the output file name/path in the parameters cell.
2. Edit the `observations` dict below (one entry per instrument).
3. Run the preview cells to check table and map.
4. Save the JSON to disk.
5. Add/update `"onshore_waves"` in the case JSON with the saved file path.
6. Build the output sites on the model mesh (isobaths + buoys) and check
   their mesh depths: the metamodel and dynamic runs use this table.

In [ ]:
from utils.config import PATHS, resolve
from utils import onshore_sites as obs
from utils import viz as plot

## Parameters

In [ ]:
from utils.config import ONSHORE_WAVES

# Output file — defaults to the path declared in the case config ("onshore_waves").
output_observations_file = ONSHORE_WAVES
output_path = resolve(output_observations_file)

print(f"output   : {output_path}")
print(f"AoI      : {resolve(PATHS['aoi_file'])}")
print(f"topobathy: {resolve(PATHS['topobathy_file'])}")

## Observations dict

Edit the dictionary below. Keys are unique `instrument_id` strings.

Required per entry:
- `lon`, `lat` — WGS84 coordinates
- `type` — role used downstream (`validation`, `waves`, `spectrum`, `calibration`, `levels`, `ignore`, …)
- `institution` — data provider (e.g. `IMOS`, `NDBC`, `SPREP`)
- `recording` — approximate coverage period as a string (e.g. `"2010-2024"`)

Optional:
- `measured_depth` — water depth from the station metadata, m positive down
  (`null` if unknown); the model uses the mesh depth (see the end of this notebook)
- `name` — human-readable station name
- `role` — overrides `type` when filtering by role in later notebooks
- any extra property (e.g. `msl` for tide gauges) is kept in the GeoJSON

In [ ]:
observations: dict[str, dict] = {
    # --- example entry: replace with your real stations ---
    "Example_Buoy": {
        "lon": 0.0,
        "lat": 0.0,
        "type": "waves",
        "institution": "TODO",
        "recording": "2000-Present",
        "measured_depth": 50.0,
        "name": "Example wave buoy",
    },
}

## Build GeoJSON and preview table

In [ ]:
feature_collection = obs.dict_to_geojson(observations)
preview_df = obs.geojson_to_dataframe(feature_collection)
preview_df

## Map

In [ ]:
plot.plot_case_map(
    obs.dict_to_gdf(observations), title="Case observations", show_satellite=True
)

## Save observations JSON (GeoJSON format)

In [ ]:
obs.save_geojson(feature_collection, output_path)

print(f"Wrote {len(feature_collection['features'])} feature(s) to:\n  {output_path}")

## Update the case JSON

After saving, add or update this key in the case JSON so downstream notebooks use your new file:

```json
"onshore_waves": "inputs/Observations/<Case>_observations.json"
```

Use the exact relative path you set in `output_observations_file`.

## Output sites on the model mesh

All output sites are built here, once, on the mesh of `simulations.model`
(`simulations.<model>.grid_file`): isobaths traced on the topobathy raster
blanked outside the mesh (`simulations.output_sites_config`), plus the buoys
just saved. Each site gets its **`mesh_depth`** (the depth the models see, used
from here on) next to its nominal depth (contour level, or the buoy's
`measured_depth`), and is kept only if it is inside the mesh and still wet at
the lowest metamodel water level (`simulations.bounds.wl[0]`, with
`simulations.min_depth`).

The table, dropped sites included, goes to `outputs/output_sites_{model}.csv`;
the metamodel cases (05) and the dynamic runs (08) read it. Rebuilding it
(`recompute=True`) invalidates runs made with the old sites.

In [ ]:
from utils.geometry import buoy_depth_check, output_sites, sites_summary

sites = output_sites(recompute=True, all_sites=True)
sites_summary(sites)

Buoys: measured depth (station metadata) against the mesh depth. A large
difference (`flag`) points to wrong coordinates, a bank or channel the mesh
smooths out, or another vertical datum.

In [ ]:
buoy_depth_check(sites)

In [ ]:
plot.plot_output_sites(sites, show_satellite=True)
plot.plot_site_depths(sites);